# Module 2: Tracing

> Self-directed module, ~20 min.

Two things in one notebook:

1. **Generate traces** — invoke the client research agent against a handful of prompts so there's something to query.
2. **Query traces** — pull runs back with `client.list_runs(...)` and the filter DSL; navigate to the project in the LangSmith UI.

With `LANGSMITH_TRACING=true`, every LLM call, tool call, and state transition is captured automatically. No code changes are required on the agent — the trace surface is opt-in at the environment level, not the application level.


## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import os
import time
from datetime import datetime, timedelta, timezone
from langsmith import Client, uuid7

from utils.config import load_active_agent, PROJECT_NAME

# Single source of truth for project name; ensure tracing is on.
os.environ["LANGSMITH_PROJECT"] = PROJECT_NAME
os.environ.setdefault("LANGSMITH_TRACING", "true")

client = Client()
agent = load_active_agent()

print(f"Project:  {PROJECT_NAME}")
print(f"Tracing:  {os.environ.get('LANGSMITH_TRACING')}")
print(f"Endpoint: {os.environ.get('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com')}")


## Warm-up: Generate traces

Five varied prompts — single-tool, profile-only, and multi-step (profile + research) — so the filter DSL has different trace shapes to work with in Part 1. Each prompt caps the search budget to keep runs fast.


In [ ]:
warmup_prompts = [
    "What's the latest on Apple (AAPL)? One paragraph, search at most once.",
    'Pull up Smith Family Office. What are their top holdings?',
    'What does TechCorp Treasury hold?',
    'Look up Acme Pension Fund, then research one of their top holdings briefly. Search at most once.',
    'Research recent news on NVDA. Search at most once.',
]

total = 0.0
for q in warmup_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].content[:120]}")

print(f"\nTotal: {total:.1f}s ({total/len(warmup_prompts):.1f}s avg)")


### Verify in LangSmith

The next cell prints the project URL. Open it and confirm five new root traces appear in the **Traces** tab. Clicking any one opens the trace tree — model spans, tool calls, and the `research-agent` subagent show up as child runs.

<img src="../images/warm_up_traces.png" alt="LangSmith Traces tab listing the warm-up runs" style="width: auto; max-height: 360px; border-radius: 8px;">


In [ ]:
try:
    project = client.read_project(project_name=PROJECT_NAME)
    print(f"Project: {project.name}")
    print(f"View traces: {project.url}")
except Exception as e:
    print(f"Could not read project (this is fine if first run): {e}")


## Part 1. Querying Traces

Once traces are landing, the SDK is one of two ways to work with them — `client.list_runs(...)` paginates over them with the same filter expression used in the UI's filter bar. The other way is just navigating the UI directly. Both are useful: programmatic for batch operations and dashboards, UI for spot-checking.


### 1.1 Pull recent traces

Useful filters on `client.list_runs(...)`:

- `project_name=` — required for cross-project queries to be reasonable.
- `start_time=` — only runs after this UTC timestamp.
- `is_root=True` — top-level traces only (skips child LLM / tool / subagent spans).
- `limit=` — cap the result set; default is 100.


In [ ]:
since = datetime.now(timezone.utc) - timedelta(hours=1)

recent_runs = list(client.list_runs(
    project_name=PROJECT_NAME,
    start_time=since,
    is_root=True,
    limit=20,
))

print(f"Found {len(recent_runs)} root run(s) in the last hour\n")
for r in recent_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds() if r.end_time else None
    print(f"- {r.id}  {r.name:25s}  latency={latency}s  error={r.error is not None}")


### 1.2 Filter DSL

The `filter` argument is a small expression language. Common patterns:

- `gt(latency, 10)` — slower than 10 seconds
- `eq(status, "error")` — failed runs
- `eq(name, "research-agent")` — runs from a specific tool or subagent
- `eq(is_root, true)` — top-level runs only (skip child spans)
- `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` — low-scored runs on a feedback key
- Combine with `and(...)` / `or(...)`

Three examples below. Each is independent — run any of them in any order. The same expressions work in the UI filter bar; the SDK is for when programmatic access matters.


#### Slow runs

Useful for spotting the multi-step traces (those that delegate to the research subagent typically take 5–15 seconds).


In [ ]:
slow_runs = list(client.list_runs(
    project_name=PROJECT_NAME,
    start_time=since,
    is_root=True,
    filter="gt(latency, 10)",
    limit=20,
))
print(f"{len(slow_runs)} slow root run(s) (>10s) in the last hour")
for r in slow_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds()
    print(f"  {r.name:25s}  {latency:.1f}s  {r.id}")


#### Errored runs

May return zero in normal operation. When debugging a flaky tool or a model rate limit, this is the first place to look.


In [ ]:
errored_runs = list(client.list_runs(
    project_name=PROJECT_NAME,
    start_time=since,
    filter='eq(status, "error")',
    limit=20,
))
print(f"{len(errored_runs)} errored run(s) in the last hour")
for r in errored_runs[:5]:
    print(f"  {r.name:25s}  {r.error[:80] if r.error else ''}")


#### By run name

Every span has a `name` (the function or tool that produced it). Filtering on `name` lets you isolate just the subagent runs, or just one specific tool, across the whole project.


In [ ]:
# Just the research-agent subagent spans — useful for inspecting what the subagent received
# TRY: change "research-agent" to "web_search" or "get_client_profile" to filter on a specific tool
subagent_runs = list(client.list_runs(
    project_name=PROJECT_NAME,
    start_time=since,
    filter='eq(name, "research-agent")',
    limit=20,
))
print(f"{len(subagent_runs)} research-agent subagent run(s) in the last hour")
for r in subagent_runs[:5]:
    print(f"  {r.id}  parent={r.parent_run_id}")


### 1.3 Navigate in the UI

The same filter DSL is available in the UI's filter bar at the top of the **Traces** tab. The UI adds inspection capabilities the SDK doesn't: trace tree visualization, side-by-side input/output panes, token cost breakdown per span, and the ability to save filters as reusable views.

For day-to-day work, expect to spend most trace inspection time in the UI; the SDK is for batch operations (export, programmatic feedback, automated quality dashboards).

<img src="../images/filter_traces.png" alt="LangSmith filter bar with a sample filter expression" style="width: auto; max-height: 320px; border-radius: 8px;">


## Recap

| Topic | API |
|---|---|
| Generate traces | `LANGSMITH_TRACING=true` + `agent.invoke(...)` |
| List runs | `client.list_runs(project_name=..., start_time=..., is_root=True)` |
| Filter — slow | `filter="gt(latency, 10)"` |
| Filter — errored | `filter='eq(status, "error")'` |
| Filter — by run name | `filter='eq(name, "research-agent")'` |

Next: `03_finding_failure_modes.ipynb` — explore your traces with Chat, Insights, and Engine.

## Additional queries to explore

Use these to build out the trace corpus further. They span four shapes, making it easier to filter for specific behaviors and surface failure modes in Module 3.

**Profile lookup only** (single `get_client_profile` call, fastest)

1. `What does Smith Family Office hold?`
2. `Pull up Acme Pension Fund and tell me about their risk profile.`
3. `What does TechCorp Treasury hold?`

**Research only** (delegates to the research subagent, no client lookup)

4. `What's the latest news on Apple (AAPL)? Search once.`
5. `Summarize Microsoft's most recent quarterly earnings.`
6. `Research recent news on the energy sector. Search at most once.`

**Multi-step** (profile + research, longer trajectories)

7. `Prep me for my meeting with Smith Family Office. Cover their top three holdings briefly.`
8. `Look up Acme Pension Fund, then research one major holding.`
9. `Write a portfolio update for TechCorp Treasury and save it to /tcorp_update.md.`

**Edge cases** (likely to surface failure modes)

10. `Look up the client 'globex-corp' and tell me about their holdings.`
11. `What was Smith Family Office's exact return last quarter?`
12. `How will AAPL stock perform next month?`